# Глубинное обучение на табличных данных

Начнем конечно с импортов различных, которые понадобятся далее. Также, многое отсюда было реализовано с помощью функций копипаста с семинаров, что мы проходили, помимо этого, семинары дали вдохновление на именно такой пайплайн действий) Стоило упомянуть

In [1]:
import pandas as pd
import numpy as np
import random

import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader #чтобы подавать данные батчами

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score #для подсчета качества

import os
import wandb

Объявим класс конфига

In [2]:
class CFG:
  project = "kolesa-cars"
  entity = "armntvs-d3v-student"
  num_epochs = 15
  train_batch_size = 64
  test_batch_size = 256
  lr = 0.001
  seed = 42
  wandb = False

In [3]:
#конфиг в словарь
def class2dict(f):
  return dict((name, getattr(f, name)) for name in dir(f) if not name.startswith('__'))

In [4]:
# вход в вандб , ключ надо вписать 1 раз
wandb.login()

wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /Users/alexander/.netrc.
wandb: Currently logged in as: armntvs-d3v (armntvs-d3v-student) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

Зафиксируем сиды

In [5]:
def seed_everything(seed): # чтобы запуски были стабильными и воспроизводимымми
    random.seed(seed) # фиксируем генератор случайных чисел
    np.random.seed(seed) # фиксируем генератор случайных чисел numpy
    torch.manual_seed(seed) # фиксируем генератор случайных чисел pytorch
    torch.cuda.manual_seed(seed) # фиксируем генератор случайных чисел для GPU

In [6]:
# https://stackoverflow.com/questions/63423463/using-pytorch-cuda-on-macbook-pro
# т.к. на macbook на их процессорах apple silicon нет cuda (только для карт nvidia), то используем альтеранативу, но если есть cuda - то используем ее
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')

device

device(type='mps')

Загрузим все предобработанные данные 

In [9]:
X_train = pd.read_csv('../data/X_train.csv')
X_test = pd.read_csv('../data/X_test.csv')
X_val = pd.read_csv('../data/X_val.csv')

y_train = pd.read_csv('../data/y_train.csv')
y_test = pd.read_csv('../data/y_test.csv')
y_val = pd.read_csv('../data/y_val.csv')


X_train.shape, X_test.shape, X_val.shape, y_train.shape, y_test.shape, y_val.shape

((4236, 570), (1060, 570), (794, 570), (4236, 1), (1060, 1), (794, 1))

Когда переводил в тензоры выдавало ошибку, потому что нужны флоат, а там видимо от ohe остались bool, переведем это все в флоат и пойдем дальше

In [10]:
X_train = X_train.astype(float)
X_test = X_test.astype(float)
X_val = X_val.astype(float)
y_train = y_train.astype(float)
y_test = y_test.astype(float)
y_val = y_val.astype(float)

Переведем все в тензоры

In [11]:
X_train_tensor =torch.FloatTensor(X_train.values)
X_test_tensor =torch.FloatTensor(X_test.values)
X_val_tensor =torch.FloatTensor(X_val.values)

y_train_tensor =torch.FloatTensor(y_train.values)
y_test_tensor =torch.FloatTensor(y_test.values)
y_val_tensor =torch.FloatTensor(y_val.values)


X_train_tensor.shape, y_train_tensor.shape

/var/folders/5n/nhsgk_xj2w1gz991pjwxs60r0000gn/T/ipykernel_56208/2613517140.py:5: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/utils/tensor_numpy.cpp:219.)
  y_train_tensor =torch.FloatTensor(y_train.values)


(torch.Size([4236, 570]), torch.Size([4236, 1]))

Теперь соединим это обратно в датасеты (train/test), чтобы передавать в модели именно все признаки с таргетом, после чего создадим dataloader чтобы подавать в модель данные батчами/частями

In [30]:
train_ds = TensorDataset(X_train_tensor,y_train_tensor)
test_ds = TensorDataset(X_test_tensor,y_test_tensor)
val_ds = TensorDataset(X_val_tensor,y_val_tensor)

train_ds

In [31]:
train_loader = DataLoader(train_ds, batch_size= CFG.train_batch_size, shuffle=True) #shuffle тут чтобы между эпохами перемешивались строки и модель не привыкала к порядку
test_loader = DataLoader(test_ds, batch_size= CFG.test_batch_size)
val_loader = DataLoader(val_ds, batch_size= CFG.test_batch_size)


In [32]:
input_size = X_train.shape[1]
input_size 

570

Сейчас построим достаточно базовую модель (вдохновление от 15 семинара). Архитектура простая - 3 полносвязных слоя и все. Далее будем строить несколько более сложных архитектур. Сначала простую чтобы понимать, помогают ли нам новые архитектуры улучшить так скажем 'базовое' качество. 

In [33]:
class Simple(nn.Module): # наследуемся от класса nn.Module
    def __init__(self, input_size):
        super().__init__()
        self.fc1 = nn.Linear(input_size, 128) # первый скрытый слой - 128 нейронов
        self.fc2 = nn.Linear(128, 64) # второй скрытый - 64
        self.fc3 = nn.Linear(64, 1) # третий  выходной- 1 прогноз
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = self.fc3(x) 
        return x

Теперь построим более сложную (чуть) архитектуру. Попробуем добавить просто больше скрытых слоев и добавим еще больше нейронов 

Но, при этом, есильно выше риск переобучения, поэтому будем все это сравнивать на качестве теста

In [34]:
class Deep(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 512) #теперь начинаем уменьшение размерности с 512 
        self.fc2 = nn.Linear(512, 256)
        self.fc3 = nn.Linear(256, 128)
        self.fc4 = nn.Linear(128, 64)
        self.fc5 = nn.Linear(64, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = torch.relu(self.fc2(x))
        x = torch.relu(self.fc3(x))
        x = torch.relu(self.fc4(x))
        x = self.fc5(x)
        return x

И, добавим третью модель, тут мы добавим еще дропаут и батчнорм. Первый будет бороться с переобучением за счет отключения определенного количества нейронов во время обучения. Батчнорм будет просто стабилизировать значения внутри срктытых слоев

In [35]:
class Regularized(nn.Module):
    def __init__(self, input_size):
        super().__init__()


        self.net = nn.Sequential(nn.Linear(input_size, 256), nn.BatchNorm1d(256), nn.ReLU(), nn.Dropout(0.05), #1 скрытый 
                                 nn.Linear(256, 128), nn.BatchNorm1d(128), nn.ReLU(),nn.Dropout(0.05),  #2 скрытый
                                 nn.Linear(128, 64), nn.ReLU(), nn.Linear(64, 1))    #3 скрытый и 4 выходной
        #sequential тут потому что во первых мы так работали на семинарах, а во-вторых потому что слои и операции в них просто идут по порядку
        #без сложностей , и следующая функция красивая в 2 строки получается )))

    def forward(self, x):
        return self.net(x)

Мы решили добавить еще одну полнсвязную сеть с residual connection. Его преимущество в том, что он сохраняет вход блока и добавляет к выходу. То есь, это хорошо потому что мы по идее не теряем юзфул информацию когда проходим несколько слоев. В нашем случае, это полезно потому что после ohe у нас стало много признаков, и там есть важные признаки ,которые residual mlp поможет учитывать в каких то сложных комбинациях

Сайты, в том числе откуда и вдохновение на код:
- https://discuss.pytorch.org/t/how-to-use-residual-learning-applied-to-fully-connected-networks/98708/5
- https://stackoverflow.com/questions/60817390/implementing-a-simple-resnet-block-with-pytorch

In [36]:
class ResBlock(nn.Module):
    def __init__(self, hidden_size):
        super().__init__()
        
        self.fc1 = nn.Linear(hidden_size, hidden_size)
        self.fc2 = nn.Linear(hidden_size, hidden_size)
        
    def forward(self, x):
        residual = x  # сохраняем вход блока
        x = torch.relu(self.fc1(x))
        x = self.fc2(x)
        x = x + residual  # прибавляем старый x к новому x
        x = torch.relu(x)
        
        return x

In [37]:
class ResMLP(nn.Module):
    def __init__(self, input_size):
        super().__init__()
        
        self.fc1 = nn.Linear(input_size, 256)
        self.block1 = ResBlock(256)
        self.block2 = ResBlock(256)
        self.fc2 = nn.Linear(256, 1)
        
    def forward(self, x):
        x = torch.relu(self.fc1(x))
        x = self.block1(x)
        x = self.block2(x)
        x = self.fc2(x)
        return x

Далее, переходим к следующей части , пока просто зададим функцию потерь и потом напишем фукнцию обучения

In [38]:
criterion = nn.MSELoss()

In [40]:
from tqdm import tqdm

def train(model, device, train_loader, optimizer, criterion, epoch, WANDB): 
    model.train()
    train_loss = 0
    
    for data, target in tqdm(train_loader):
        data = data.to(device)
        target = target.to(device)
        
        optimizer.zero_grad() #обнуяем градиенты
        
        output = model(data) #прямой проход
        loss = criterion(output, target) #считаем ошибку
        
        loss.backward() #обратынй проход
        optimizer.step() #шаг оптимизатором
        train_loss += loss.item()
    
    train_loss = train_loss / len(train_loader)
    
    tqdm.write('\nTrain set: Average loss: {:.4f}'.format(train_loss)) #как в семе, но чуть поправленная, потому что там для картинок

    if WANDB:
        wandb.log({'epoch': epoch,
                   'train_loss': train_loss})
        
    return train_loss

Теперь функция тестирования, оч похожа на сем, но там картинки

In [41]:
def eval(model, device, test_loader, criterion, epoch, WANDB, stage='Test'):
    model.eval()  
    
    test_loss = 0
    preds = []
    true = []
    
    # показываем, что обучения нет и градиенты не обновляются
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            
            output = model(data)
            test_loss += criterion(output, target).item()
            
            preds.append(output.cpu())
            true.append(target.cpu())
    
    test_loss = test_loss / len(test_loader)
    
    preds = torch.cat(preds).numpy().ravel() #тут мы клеим прогнозы в одномерный массив нампай 
    true = torch.cat(true).numpy().ravel()
    
    preds = np.expm1(preds) #возвращаем логарифмированные числа в первоначальное 
    true = np.expm1(true)
    
    mae = mean_absolute_error(true, preds)
    rmse = np.sqrt(mean_squared_error(true, preds))
    r2 = r2_score(true, preds)
    
    tqdm.write('{} set: Average loss: {:.4f}, MAE: {:.2f}, RMSE: {:.2f}, R2: {:.4f}'.format(
       stage, test_loss, mae, rmse, r2))
    
    if WANDB:
        wandb.log({'epoch': epoch,
                   f'{stage.lower()}_loss': test_loss,
                   f'{stage.lower()}_mae': mae,
                   f'{stage.lower()}_rmse': rmse,
                   f'{stage.lower()}_r2': r2})
    
    return test_loss, mae, rmse, r2

Теперь нужна функция запуска наших экспериментов, и после этого можно будет лицезреть качество выстроенных нами моделей))

In [42]:
# основная функция для экспериментов
def main(model, model_name):

    if CFG.wandb:
        wandb.init(project=CFG.project, entity=CFG.entity, name=model_name, reinit=True, config=class2dict(CFG)) #reinit чтобы каждый экспер записывался как новый
        # параметры архитектуры https://docs.wandb.ai/guides/track/config
        wandb.config.update({'model': str(model)})
    seed_everything(CFG.seed)  # фиксируем сиды
    
    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=CFG.lr)
    
    train_losses = []
    val_losses = []
    
    if CFG.wandb:
        wandb.watch(model, log='all') # логируем все (метрики, лоссы, градиенты)


    for epoch in range(1, CFG.num_epochs + 1):
        print('\nEpoch:', epoch)
        train_loss = train(model, device, train_loader, optimizer, criterion, epoch, CFG.wandb)
        val_loss, val_mae, val_rmse, val_r2 = eval(model, device, val_loader, criterion, epoch, CFG.wandb, stage='Val')
        train_losses.append(train_loss)
        val_losses.append(val_loss)

    print('Training is ended!')
    
    test_loss, test_mae, test_rmse, test_r2 = eval(model, device, test_loader, criterion, CFG.num_epochs, CFG.wandb, stage='Test')

    if CFG.wandb:
        #сохраняем модель https://docs.wandb.ai/guides/artifacts
        os.makedirs('../data/models', exist_ok=True)
        torch.save(model.state_dict(), f'../data/models/{model_name}.pt')
        artifact = wandb.Artifact(model_name, type='model')
        artifact.add_file(f'../data/models/{model_name}.pt')
        wandb.log_artifact(artifact)
        wandb.finish()

    return model, train_losses, val_losses, test_loss, test_mae, test_rmse, test_r2

In [43]:
#https://docs.wandb.ai/guides/artifacts
#выполнить один раз, далее закомментировать если повторно прогонять
if CFG.wandb:
    wandb.init(project=CFG.project, entity=CFG.entity, name='dataset-tabular', reinit=True)
    art = wandb.Artifact('kolesa-tabular', type='dataset')
    for f in ['X_train.csv', 'X_val.csv', 'X_test.csv', 'y_train.csv', 'y_val.csv', 'y_test.csv']:
        art.add_file(f'../data/{f}')
    wandb.log_artifact(art)
    wandb.finish()

In [44]:
seed_everything(CFG.seed)
simple_model, simple_train_losses, simple_val_losses, simple_test_loss, simple_mae, simple_rmse, simple_r2 = main(Simple(input_size),'Simple')


Epoch: 1


100%|██████████| 67/67 [00:01<00:00, 40.44it/s]



Train set: Average loss: 154.6181
Val set: Average loss: 6.2640, MAE: 164280032.00, RMSE: 436118459.81, R2: -1868.1062

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 621.04it/s]



Train set: Average loss: 3.1786
Val set: Average loss: 1.8983, MAE: 12003065.00, RMSE: 22589887.47, R2: -4.0148

Epoch: 3


...


Train set: Average loss: 0.0873
Val set: Average loss: 0.2082, MAE: 2767019.25, RMSE: 5942715.52, R2: 0.6529

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 583.70it/s]



Train set: Average loss: 0.0755
Val set: Average loss: 0.2016, MAE: 2534696.25, RMSE: 5293456.65, R2: 0.7246

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 565.46it/s]



Train set: Average loss: 0.0675
Val set: Average loss: 0.1926, MAE: 2451405.50, RMSE: 5266931.43, R2: 0.7274
Training is ended!
Test set: Average loss: 0.2489, MAE: 2573330.25, RMSE: 5338706.86, R2: 0.7233


In [45]:
seed_everything(CFG.seed)
deep_model, deep_train_losses, deep_val_losses, deep_test_loss, deep_mae, deep_rmse, deep_r2 = main(Deep(input_size), 'Deep')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 120.76it/s]



Train set: Average loss: 69.2995
Val set: Average loss: 2.0554, MAE: 33375550.00, RMSE: 147089620.94, R2: -211.6127

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 486.25it/s]



Train set: Average loss: 0.9805
Val set: Average loss: 0.7822, MAE: 7414953.00, RMSE: 28864918.57, R2: -7.1878

Epoch: 3


...


Train set: Average loss: 0.0504
Val set: Average loss: 0.1656, MAE: 2663137.75, RMSE: 5240137.88, R2: 0.7302

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 347.98it/s]



Train set: Average loss: 0.0501
Val set: Average loss: 0.1703, MAE: 2228375.50, RMSE: 4733201.61, R2: 0.7798

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 352.12it/s]



Train set: Average loss: 0.0457
Val set: Average loss: 0.1721, MAE: 2210639.00, RMSE: 4230007.79, R2: 0.8242
Training is ended!
Test set: Average loss: 0.2174, MAE: 2350698.25, RMSE: 4476950.95, R2: 0.8054


In [46]:
seed_everything(CFG.seed)
regularized_model, regularized_train_losses, regularized_val_losses, regularized_test_loss, regularized_mae, regularized_rmse, regularized_r2 = main(
    Regularized(input_size),'Regularized')


Epoch: 1


  0%|          | 0/67 [00:00<?, ?it/s]

100%|██████████| 67/67 [00:00<00:00, 70.00it/s]



Train set: Average loss: 113.1711
Val set: Average loss: 4.3403, MAE: 9058258.00, RMSE: 12836117.05, R2: -0.6192

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 433.80it/s]


...


Train set: Average loss: 0.5363
Val set: Average loss: 0.2668, MAE: 4326813.50, RMSE: 9462171.81, R2: 0.1202

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 268.01it/s]



Train set: Average loss: 0.5583
Val set: Average loss: 0.2910, MAE: 4314310.50, RMSE: 6995380.73, R2: 0.5191

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 337.31it/s]



Train set: Average loss: 0.4687
Val set: Average loss: 0.4360, MAE: 4917106.00, RMSE: 7800101.00, R2: 0.4021
Training is ended!
Test set: Average loss: 0.4182, MAE: 5057628.00, RMSE: 7938428.16, R2: 0.3881


In [49]:
seed_everything(CFG.seed)
res_model, res_train_losses, res_val_losses, res_test_loss, res_mae, res_rmse, res_r2 = main(ResMLP(input_size), 'ResMLP')


Epoch: 1


100%|██████████| 67/67 [00:00<00:00, 135.26it/s]



Train set: Average loss: 41.0732
Val set: Average loss: 0.9354, MAE: 9867605.00, RMSE: 36000961.04, R2: -11.7366

Epoch: 2


100%|██████████| 67/67 [00:00<00:00, 407.57it/s]



Train set: Average loss: 0.5792
Val set: Average loss: 0.4418, MAE: 5767093.00, RMSE: 21042287.00, R2: -3.3512

Epoch: 3


...


Train set: Average loss: 0.0407
Val set: Average loss: 0.1289, MAE: 2944112.25, RMSE: 5402411.47, R2: 0.7132

Epoch: 14


100%|██████████| 67/67 [00:00<00:00, 435.91it/s]



Train set: Average loss: 0.0401
Val set: Average loss: 0.1198, MAE: 2257676.50, RMSE: 4089095.62, R2: 0.8357

Epoch: 15


100%|██████████| 67/67 [00:00<00:00, 444.50it/s]


Train set: Average loss: 0.0423
Val set: Average loss: 0.1169, MAE: 2097884.50, RMSE: 4004236.22, R2: 0.8424
Training is ended!
Test set: Average loss: 0.1499, MAE: 2369907.50, RMSE: 4301483.10, R2: 0.8204


In [50]:
res = pd.DataFrame([{'model': 'Simple', 'test_loss': simple_test_loss, 'MAE': simple_mae, 'RMSE': simple_rmse, 'R2': simple_r2},
                        {'model': 'Deep', 'test_loss': deep_test_loss, 'MAE': deep_mae, 'RMSE': deep_rmse, 'R2': deep_r2},
                        {'model': 'Regularized', 'test_loss': regularized_test_loss, 'MAE': regularized_mae, 'RMSE': regularized_rmse, 'R2': regularized_r2},
                        {'model': 'ResMLP', 'test_loss': res_test_loss, 'MAE': res_mae, 'RMSE': res_rmse, 'R2': res_r2}])

res

,model,test_loss,MAE,RMSE,R2
0,Simple,0.248894,2573330.25,5.338707e+06,0.723267
1,Deep,0.217364,2350698.25,4.476951e+06,0.805396
2,Regularized,0.418166,5057628.00,7.938428e+06,0.388133
3,ResMLP,0.149928,2369907.50,4.301483e+06,0.820351


In [51]:
res.sort_values('MAE')

,model,test_loss,MAE,RMSE,R2
1,Deep,0.217364,2350698.25,4.476951e+06,0.805396
3,ResMLP,0.149928,2369907.50,4.301483e+06,0.820351
0,Simple,0.248894,2573330.25,5.338707e+06,0.723267
2,Regularized,0.418166,5057628.00,7.938428e+06,0.388133


### Сравнение моделей и выводы

По ходу работы мы обучили 4 модели, от простой базовой полносвязной сети до полносвязной с residual блоками. Основная метрика - считаем MAE, средняя ошибка прогноза цены в тенге. 

По результатам: лучшей стала модель Deep с показателями MAE - примерно 2,34 млн тенге, R2 - примерно 0,83, то есть модель лучше остальных отличает цены и в среднем меньше ошибается. Базовая модель(Simple) и модель с ResMLP также дают хороший результат. Худший результат показала модель Regularized, возможно dropout для нашего датасета мешала модели запоминать какие-то важные зависимости в данных или еще что-то, но любые манипуляции с весами не помогали. 